# 02b. Fine-Tuning Incremental: Especialização em Placas Brasileiras

Este notebook realiza o **Treinamento Incremental** do detector YOLOv8 a partir do modelo treinado no notebook `02`, especializando-o no dataset de **placas veiculares brasileiras** (Jhonathas Moura / IFRN).

### Por que este notebook existe?
- O modelo base aprendeu a geometria geral de 8.800 placas no notebook `02`.
- Aqui, usamos **taxa de aprendizado controlada (`lr0=0.001`)** para refinar a atenção da rede nas características de veículos e placas brasileiras (padronização Mercosul e cinza antiga, iluminações reais do país) sem sofrer *esquecimento catastrófico*.
- Atende ao experimento **EXP-07** do planejamento do projeto.

--- 
## 1. Instalação e Checagem da GPU
*Certifique-se de que a GPU está ativa em: `Ambiente de Execução` -> `Alterar tipo de ambiente de execução` -> `T4 GPU`.*

In [ ]:
!pip install -q ultralytics onnx onnxruntime

import torch
print('Versão do PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('✅ GPU detectada:', torch.cuda.get_device_name(0))
else:
    print('⚠️ ATENÇÃO: Nenhuma GPU detectada. Ative a GPU no menu Ambiente de Execução!')

--- 
## 2. Carregar o Checkpoint Anterior (`best.pt`)
Se você estiver executando no mesmo ambiente do notebook `02`, ele detecta o modelo automaticamente. Caso contrário, permite fazer o upload do `best.pt` do seu computador.

In [ ]:
from pathlib import Path
from google.colab import files

# Tenta localizar o best.pt da sessão atual
candidatos = list(Path('/content').glob('**/treino_placas/weights/best.pt'))
if not candidatos:
    candidatos = list(Path('/content').glob('**/best.pt'))

if candidatos:
    caminho_base = str(candidatos[0])
    print('✅ Checkpoint do treino anterior encontrado:', caminho_base)
else:
    print('Checkpoint anterior não encontrado no disco local do Colab.')
    print('Por favor, faça o upload do arquivo best.pt gerado no primeiro treino:')
    upload = files.upload()
    caminho_base = list(upload.keys())[0]
    print('✅ Checkpoint carregado com sucesso:', caminho_base)

--- 
## 3. Download e Organização do Dataset Brasileiro (400 Imagens)
Baixa o arquivo compactado diretamente do repositório público do IFRN e organiza a estrutura de diretórios para o padrão YOLOv8.

In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path

print('Baixando dataset.zip do GitHub (jhonathasmoura/Deteccao_Placas_Veiculos_YOLOv4)...')
url = 'https://raw.githubusercontent.com/jhonathasmoura/Deteccao_Placas_Veiculos_YOLOv4/main/dataset.zip'
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})

with urllib.request.urlopen(req) as resp:
    zip_bytes = resp.read()

print(f'Download concluído: {len(zip_bytes) / (1024 * 1024):.2f} MB')

# Extrai arquivos brutos
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    z.extractall('/content/dataset_brasil_raw')

# Estruturação nas pastas YOLOv8
base_dest = Path('/content/dataset_brasil')
mapeamento = [
    ('train', 'train'),
    ('valid', 'val'),
    ('test', 'test')
]

for pasta_origem, split_dest in mapeamento:
    img_dir = base_dest / 'images' / split_dest
    lbl_dir = base_dest / 'labels' / split_dest
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    
    origem = Path(f'/content/dataset_brasil_raw/{pasta_origem}')
    for arq in origem.glob('*.*'):
        if arq.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            arq.replace(img_dir / arq.name)
        elif arq.suffix.lower() == '.txt':
            arq.replace(lbl_dir / arq.name)

# Configuração placas_brasil.yaml
yaml_brasil = '''
path: /content/dataset_brasil
train: images/train
val: images/val
test: images/test

names:
  0: placa
'''

with open('/content/placas_brasil.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_brasil.strip())

print('✅ Dataset brasileiro estruturado com sucesso!')

--- 
## 4. Fine-Tuning Incremental
Iniciamos a partir do `best.pt` anterior com `lr0=0.001` (taxa suave para ajuste fino).

In [ ]:
from ultralytics import YOLO

modelo_incremental = YOLO(caminho_base)

resultados = modelo_incremental.train(
    data='/content/placas_brasil.yaml',
    epochs=25,
    imgsz=640,
    batch=16,
    lr0=0.001,
    patience=10,
    project='runs/detect',
    name='treino_brasil',
    verbose=True
)

--- 
## 5. Avaliação das Métricas em Placas Brasileiras

In [ ]:
from pathlib import Path
from ultralytics import YOLO

candidato_final = list(Path('/content').glob('**/treino_brasil*/weights/best.pt'))[0]
modelo_final = YOLO(str(candidato_final))

metricas = modelo_final.val(data='/content/placas_brasil.yaml', split='test')

print('=' * 55)
print('MÉTRICAS DO MODELO ESPECIALIZADO NO BRASIL')
print('=' * 55)
print(f'mAP@0.5:      {metricas.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metricas.box.map:.4f}')
print(f'Precisão:     {metricas.box.mp:.4f}')
print(f'Recall:       {metricas.box.mr:.4f}')
print('=' * 55)

--- 
## 6. Exportação para Formato ONNX

In [ ]:
caminho_onnx_brasil = modelo_final.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True
)
print('✅ Modelo ONNX especializado exportado em:', caminho_onnx_brasil)

--- 
## 7. Verificação com onnxruntime em CPU

In [ ]:
import onnxruntime as ort

sessao = ort.InferenceSession(caminho_onnx_brasil, providers=['CPUExecutionProvider'])
entrada = sessao.get_inputs()[0]
saida = sessao.get_outputs()[0]

print('Entrada ONNX:', entrada.name, entrada.shape)
print('Saída ONNX:  ', saida.name, saida.shape)
print('✅ Modelo verificado com sucesso!')

--- 
## 8. Download do Modelo Final para a Aplicação Desktop

In [ ]:
from google.colab import files

print(f'Iniciando download de: {caminho_onnx_brasil}')
files.download(caminho_onnx_brasil)